## Comparing individual vs stacked richness. 

Overview: This code examines if the individual vs stacked richness agree. Whether to shift from one to another one needs to apply some kind of Eddington bias correction. Whether the weighting agrees for individual vs stacked. 

In [1]:
from scipy.stats import kde
import h5py
import astropy.io.fits as fits
import csv
import pandas as pd
import numpy as np
import tables
import pickle
import os
from astropy.table import Table
from astropy.coordinates import SkyCoord
from tqdm import tqdm
from astropy.io import ascii
import os
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import incredible as cr
from scipy.special import erf
from scipy import stats
import scipy.optimize as opt
from scipy import stats
import scipy.optimize as opt
import emcee
import tqdm
import pickle
from astropy import table
from astropy.table import Table, join, unique
from specutils import SpectralRegion
from scipy.interpolate import BSpline, make_interp_spline, UnivariateSpline

In [2]:
from astropy.cosmology import FlatLambdaCDM
from scipy.interpolate import interp1d
import numpy as np
import astropy.units as u
import astropy.cosmology.units as cu
from astropy.cosmology import Planck18
from astropy.cosmology import z_at_value
cosmo = Planck18

import sys
sys.path.append("/global/homes/z/zzhang13/DESI")
from setup import *
## Functions for computing spectroscopic richness and profiles
from tools.projection_functions import *

In [3]:
import matplotlib.patches as patches
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt


plot_params = {

'axes.linewidth': 1.25,
'xtick.major.width': 1, 'ytick.major.width': 1,
'xtick.minor.width': 1, 'ytick.minor.width': 1,
'xtick.major.size': 12, 'ytick.major.size': 12,
'xtick.labelsize': 12, 'ytick.labelsize': 12,
'xtick.direction': 'in', 'ytick.direction': 'in',
'xtick.major.pad': 6, 'xtick.minor.pad': 6,
'figure.constrained_layout.use': True,
#'text.usetex': True,
'font.family': 'Serif',
'font.size': 14,
'legend.fontsize': 11
    
}
plt.rcParams.update(plot_params)

## Import Catalogs

* The fluxes are corrected for MW Extinction.
* (!!) Not sure if corrected for K-correction. If not I may have to apply one myself. 

In [4]:
with open(data_dir()+'bgs_clus_RM_gal_matched.pickle', 'rb') as handle:
    bgs_matched = pickle.load(handle)

bgs_matched.columns

<TableColumns names=('TARGETID','RA_BGS','DEC_BGS','Z_BGS','WEIGHT','flux_g_dered','flux_r_dered','flux_z_dered','flux_w1_dered','flux_w2_dered','ID','LAMBDA','Z_LAMBDA','R_LAMBDA','Z_SPEC_x','RA_x','DEC_x','MODEL_MAG_R_x','MODEL_MAGERR_R_x','RM_gal_flag','Z_SPEC_y','RA_y','DEC_y','R','P','MODEL_MAG_R_y','MODEL_MAGERR_R_y','central_flag','geoFrac')>

In [5]:
fcols = ['g','r','z','w1','w2']
for col in fcols:
    #bgs_matched['flux_'+col.lower()+'_dered'] = bgs_matched['FLUX_'+col]/data['MW_TRANSMISSION_'+col]
    bgs_matched['r_dered'] = 22.5 - 2.5*np.log10(bgs_matched['flux_r_dered'])
    bgs_matched['g_dered'] = 22.5 - 2.5*np.log10(bgs_matched['flux_g_dered'])
    bgs_matched['gmr'] = bgs_matched['g_dered']-bgs_matched['r_dered']

In [6]:
#binGap = 1e-5
wide_bin_1 = np.linspace(-0.1,-0.005,21, endpoint=False)
small_bin = np.linspace(-0.005,0.005,21, endpoint=False)
micro_bin = np.linspace(-0.005,0.005,31)
wide_bin_2 = np.linspace(0.005,0.1,21, endpoint=False)


binBoundaries = np.hstack((wide_bin_1, small_bin))
binBoundaries = np.hstack((binBoundaries, wide_bin_2))
binCent = np.asarray([(binBoundaries[i] + binBoundaries[i+1])/2 for i in range(len(binBoundaries)-1)])
binCent_micro = np.asarray([(micro_bin[i] + micro_bin[i+1])/2 for i in range(len(micro_bin)-1)])
bin_width = np.asarray([binBoundaries[i+1]-binBoundaries[i] for i in range(len(binBoundaries[:-1]))])

## None overlapping bins
assert len(set(binCent)) == len(binCent), "Overlapping bins"
assert len(set(binBoundaries)) == len(binBoundaries), "Overlapping bins"

## Binning

In [7]:
#Bin by richness
lmda_bins = [[20,22],[22,25],[25,30],[30,40],[50,200]] #upper limit must match lower limit of next bin
## Bin by redshift
z_bins = [[0.1,0.2],[0.2,0.3],[0.3,0.4]]

lmda_bin_edges = np.logspace(np.log10(20), np.log10(100),11)
lmda_bins = [[lmda_bin_edges[i], lmda_bin_edges[i+1]] for i in range(len(lmda_bin_edges)-1)]

##Create an absolute magnitude column
cosmo = Planck18
h = 0.67
M_r = bgs_matched['r_dered']-cosmo.distmod(bgs_matched['Z_BGS']).value - 5*np.log10(h)
bgs_matched['M_r'] = M_r ##comoving M with h-scaling

##Cuts
bgs_matched = bgs_matched[np.where(bgs_matched['r_dered'] < 19.5)]
#bgs_matched = bgs_matched[np.where(bgs_matched['Z_BGS']< 0.3)]
#bgs_matched = bgs_matched[np.where(bgs_matched['M_r'] < 22)]
#bgs_matched = bgs_matched[np.where(bgs_matched['LAMBDA'] > 100)]

In [8]:
#Assigning spectrospic richnesses to individual cluster. 
from scipy.stats import gaussian_kde
import matplotlib.colors as colors

rmTable = unique(bgs_matched, keys='ID')
ID_list, lambda_tot_list, lambda_true_list  = calc_specRichness_individual(binBoundaries, binCent, bgs_matched)
lambda_rm = [rm_lambda for i, rm_lambda in enumerate(rmTable['LAMBDA']) if rmTable['ID'][i] in ID_list] 
rm_id = [rmTable['ID'][i] for i, rm_lambda in enumerate(rmTable['LAMBDA']) if rmTable['ID'][i] in ID_list] 
ind = [i for i, rm_lambda in enumerate(rmTable['LAMBDA']) if rmTable['ID'][i] in ID_list]

lambda_tot_list = np.asarray(lambda_tot_list)
lambda_true_list = np.asarray(lambda_true_list)
lambda_rm = np.asarray(lambda_rm)

##Filtering out the clusters with no spectrospic richness information and assigning spectroscopic information
rmTable = rmTable[ind]
rmTable['lambda_true'] = lambda_true_list
rmTable['lambda_tot'] = lambda_tot_list
rm_df = rmTable.to_pandas()

## Run Diagnostics

Run diagnostics between stacked and individual. 

In [9]:
from astropy.table import Table
import numpy as np
from tools.richness_diagnostics import run_diagnostics

table = bgs_matched

bin_edges = np.linspace(-0.08, 0.08, 81)

result, relation = run_diagnostics(
    table,
    bin_edges,
    original_module_path=tools_dir() + "projection_functions.py",
    redmapper_col="LAMBDA",   # change if your redMaPPer richness column differs
    output_dir="diagnostic_plots",
    redmapper_bins=np.logspace(np.log10(20),np.log10(200),8)
)

Wrote diagnostic plots to: /global/u1/z/zzhang13/DESI/richness_relation/diagnostic_plots
Clusters: 6696
Galaxies: 87998
Galaxies inside bin range: 67063
Fraction outside bin range: 0.2379
Clusters with finite original individual PDFs: 6539
Clusters skipped/invalid in original individual PDFs: 157
